In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# RGB-DCT local gradient T44/T46/T48 fixed trial

Run all once. Each fixed source attempts OFF, the existing no-gradient SINGLE49 reference, and LOCAL44_46_48 from the same noise and full UniPC history. The local arm recomputes the full 181-frame RGB-DCT proxy and its velocity gradient at each controlled step. Every original full MP4 is blindly scored by the frozen C>=24 rule; setup, worker, resource, gradient, and receiver failures remain in the six-slot denominator. The user performs the GPU/Colab run.


In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = 'f0951d2bd3fa666d5372b2ba1395c6924089726c'
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/RGB-DCT-Local-Gradient-V1')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / stamp
OUTPUT.mkdir(exist_ok=False)
SETUP_SLOTS_PATH = OUTPUT / 'setup_slots.json'
setup_slots = dict(
    status='GLOBAL_SETUP_NOT_COMPLETED', source_sha=SOURCE_SHA,
    fixed_denominator=dict(sources=2, mp4_score_slots=6, frames=1086),
    cases={case_id: {arm: dict(status='NOT_RUN_GLOBAL_SETUP_FAILURE', reason='SETUP_NOT_COMPLETED', attempted=False)
                     for arm in ('OFF', 'SINGLE49', 'LOCAL44_46_48')}
           for case_id in ('eval_clock_s2431', 'eval_umbrella_s2432')},
)
SETUP_SLOTS_PATH.write_text(json.dumps(setup_slots, indent=2) + '\n', encoding='utf-8')
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(status='SETUP_STARTED', source_sha=SOURCE_SHA, output_dir=str(OUTPUT), python=sys.version, executable=sys.executable), indent=2) + '\n', encoding='utf-8')
print('fresh output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys, json, traceback
SETUP_LOG = OUTPUT / 'setup.log'
def _failure_value(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='replace')
    if isinstance(value, (list, tuple)):
        return [_failure_value(item) for item in value]
    return repr(value)
def persist_global_failure(stage, exc):
    failure = dict(stage=stage, exception_class=type(exc).__name__, message=str(exc),
                   traceback=traceback.format_exc())
    for source, target in (('cmd', 'command'), ('returncode', 'return_code'),
                           ('stdout', 'stdout'), ('stderr', 'stderr'),
                           ('output', 'output')):
        if hasattr(exc, source):
            failure[target] = _failure_value(getattr(exc, source))
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        log.write('FAILURE ' + json.dumps(failure, ensure_ascii=False) + '\n')
    (OUTPUT / 'setup_failure.json').write_text(
        json.dumps(failure, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    setup_slots = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
    setup_slots['status'] = 'GLOBAL_SETUP_OR_RUN_FAILURE'
    setup_slots['failure'] = failure
    reason = stage + ':' + type(exc).__name__ + ':' + str(exc)
    for slots in setup_slots['cases'].values():
        for slot in slots.values():
            slot.update(status='NOT_RUN_GLOBAL_SETUP_FAILURE', reason=reason, attempted=True)
    SETUP_SLOTS_PATH.write_text(
        json.dumps(setup_slots, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    (OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(
        status='SETUP_OR_RUN_FAILED', source_sha=SOURCE_SHA, output_dir=str(OUTPUT),
        failure=failure), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
def guarded_setup(stage, source):
    try:
        exec(compile(source, '<fixed-' + stage.lower() + '>', 'exec'), globals(), globals())
    except Exception as exc:
        persist_global_failure(stage, exc)
        raise
def logged_run(command, *, cwd=None, env=None, check=True):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)
guarded_setup('INSTALL', 'import importlib.metadata, subprocess, sys\nprint(\'Python:\', sys.version, flush=True)\nprint(\'Executable:\', sys.executable, flush=True)\nlogged_run([sys.executable, \'-m\', \'pip\', \'--version\'], check=True)\nlogged_run([\'apt-get\', \'update\', \'-qq\'], check=True)\nlogged_run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'ffmpeg\'], check=True)\ndef version(name):\n    try:\n        return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        return None\nprint(\'torch before install:\', version(\'torch\'), flush=True)\nif (version(\'torch\') or \'\').split(\'+\', 1)[0] != \'2.11.0\':\n    logged_run([sys.executable, \'-m\', \'pip\', \'install\', \'torch==2.11.0\', \'torchvision\', \'--index-url\', \'https://download.pytorch.org/whl/cu128\'], check=True)\nlogged_run([sys.executable, \'-m\', \'pip\', \'install\', \'diffusers==0.40.0\', \'transformers\', \'accelerate\', \'ftfy\', \'sentencepiece\', \'safetensors\', \'huggingface_hub\', \'numpy\', \'Pillow\'], check=True)\ncheck_code = """\nimport importlib.metadata, sys, torch, diffusers\nfrom diffusers import WanPipeline, AutoencoderKLWan\nprint(\'Fresh process Python:\', sys.version, flush=True)\nprint(\'Fresh process executable:\', sys.executable, flush=True)\nfor name in (\'torch\', \'torchvision\', \'diffusers\', \'transformers\', \'accelerate\', \'ftfy\', \'sentencepiece\', \'safetensors\', \'huggingface_hub\', \'numpy\', \'Pillow\'):\n    try:\n        value = importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        value = None\n    print(name + \':\', value, flush=True)\nassert str(torch.__version__).split(\'+\', 1)[0] == \'2.11.0\', torch.__version__\nassert torch.cuda.is_available(), \'CUDA torch required\'\nassert diffusers.__version__ == \'0.40.0\', diffusers.__version__\n"""\nlogged_run([sys.executable, "-u", "-c", check_code], check=True)\n')


In [ ]:
guarded_setup('SOURCE_CHECKOUT', "REPO = Path('/content/SC-SSTW-RGB-DCT-LOCAL-GRADIENT-' + stamp)\nlogged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])\nlogged_run(['git', '-C', str(REPO), 'fetch', 'origin', 'dev/rgb-dct-local-gradient-v1'])\nlogged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])\nactual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()\nif actual != SOURCE_SHA:\n    raise RuntimeError('immutable source SHA readback mismatch')\n(OUTPUT / 'source_receipt.json').write_text(json.dumps(dict(expected_sha=SOURCE_SHA, actual_sha=actual, repo=str(REPO)), indent=2) + '\\n', encoding='utf-8')\nprint('source commit:', actual, flush=True)\n")


In [ ]:
guarded_setup('ENVIRONMENT', "import shutil, torch, numpy, diffusers\nfrom diffusers import WanPipeline, AutoencoderKLWan\nenvironment_receipt = dict(\n    ffmpeg=shutil.which('ffmpeg'), ffprobe=shutil.which('ffprobe'),\n    python=sys.version, torch=torch.__version__, torch_cuda_runtime=torch.version.cuda,\n    numpy=numpy.__version__, diffusers=diffusers.__version__,\n    cuda_available=torch.cuda.is_available(),\n    device=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),\n    required_api=dict(autograd_grad=callable(torch.autograd.grad),\n                      wan_pipeline=WanPipeline is not None, wan_vae=AutoencoderKLWan is not None),\n)\n(OUTPUT / 'environment_receipt.json').write_text(json.dumps(environment_receipt, indent=2) + '\\n', encoding='utf-8')\nif (not environment_receipt['ffmpeg'] or not environment_receipt['ffprobe']\n        or not environment_receipt['cuda_available']\n        or str(torch.__version__).split('+', 1)[0] != '2.11.0'\n        or diffusers.__version__ != '0.40.0'\n        or not all(environment_receipt['required_api'].values())):\n    setup_slots = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))\n    for slots in setup_slots['cases'].values():\n        for slot in slots.values():\n            slot.update(reason='ENVIRONMENT_VALIDATION_FAILED', attempted=True)\n    SETUP_SLOTS_PATH.write_text(json.dumps(setup_slots, indent=2) + '\\n', encoding='utf-8')\n    raise RuntimeError('fixed Wan environment validation failed')\n(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(status='SETUP_COMPLETE', source_sha=SOURCE_SHA, output_dir=str(OUTPUT), python=sys.version, executable=sys.executable), indent=2) + '\\n', encoding='utf-8')\nprint('fixed Wan environment:', environment_receipt, flush=True)\n")


In [ ]:
guarded_setup('RUN', "import os\nenv = os.environ.copy()\nenv['PYTHONUNBUFFERED'] = '1'\ncommand = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.rgb_dct_local_gradient_run', '--output', str(OUTPUT)]\ncompleted = logged_run(command, cwd=REPO, env=env, check=False)\nRESULT_PATH = OUTPUT / 'result.json'\n(OUTPUT / 'execution_receipt.json').write_text(json.dumps(dict(command=command, returncode=completed.returncode, result_path=str(RESULT_PATH)), indent=2) + '\\n', encoding='utf-8')\nif not RESULT_PATH.exists():\n    raise FileNotFoundError('runner produced no retained result.json')\nsetup_slots = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))\nsetup_slots['status'] = 'SUPERSEDED_BY_RESULT_JSON'\nsetup_slots['result_path'] = str(RESULT_PATH)\nSETUP_SLOTS_PATH.write_text(json.dumps(setup_slots, indent=2) + '\\n', encoding='utf-8')\n")


In [ ]:
guarded_setup('RESULT_PRESENTATION', "result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))\nif result['fixed_denominator'] != {'sources': 2, 'mp4_score_slots': 6, 'frames': 1086}:\n    raise RuntimeError('fixed denominator mismatch')\nprint('status:', result['status'], flush=True)\nprint('fixed C rule:', result['decision_rule'], flush=True)\nprint('attempted/scored/invalid/pending:', result['attempted_media_slots'], result['scored_media_slots'], result['invalid_media_slots'], result['pending_media_slots'], flush=True)\nfor case_id, case in result['cases'].items():\n    for arm, row in case['slots'].items():\n        print(case_id, arm, row['status'], row['positive_groups'], row['decision'], row['reason'], flush=True)\nprint('full result:', RESULT_PATH, flush=True)\n")
